In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen3-0.6B"

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype="auto", device_map="auto"
)

# prepare the model input
prompt = "Give me a short introduction to large language model."
messages = [{"role": "user", "content": prompt}]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True,  # Switches between thinking and non-thinking modes. Default is True.
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# conduct text completion
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=32768,
    do_sample=False,
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]) :].tolist()

# parsing thinking content
try:
    # rindex finding 151668 (</think>)
    index = len(output_ids) - output_ids[::-1].index(151668)
except ValueError:
    index = 0

thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip(
    "\n"
)
content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

print("thinking content:", thinking_content)
print("content:", content)
pass

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


thinking content: <think>
Okay, the user wants a short introduction to a large language model. Let me start by recalling what I know about LLMs. They're big language models, right? So I should mention their ability to understand and generate text. Maybe start with the basics: they're trained on massive datasets, so they can learn a lot. Then talk about their capabilities, like understanding context, generating coherent responses, and being able to handle various tasks. Also, mention that they're not just text generators but can perform tasks like writing, answering questions, and even creating content. I should keep it concise but cover the key points. Let me check if I'm missing anything. Oh, maybe include that they can be used in different applications. Alright, that should do it.
</think>
content: A large language model (LLM) is a type of artificial intelligence designed to understand and generate human language. These models are trained on vast datasets to learn patterns and contex

In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# --- 1. Загрузка модели и токенизатора (без изменений) ---
model_name = "Qwen/Qwen3-0.6B"

# Загружаем токенизатор и модель
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype="auto", device_map="auto"
)

# --- 2. Подготовка входных данных (без изменений) ---
prompt = "Give me a short introduction to large language model."
messages = [{"role": "user", "content": prompt}]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,  # Включаем режим "мышления"
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# --- 3. Генерация текста с использованием цикла for и greedy decoding ---

# Начальные ID токенов (из нашего промпта)
input_ids = model_inputs.input_ids
# attention_mask также важна, чтобы модель знала, на какие токены обращать внимание
attention_mask = model_inputs.attention_mask

# Список для хранения сгенерированных ID токенов
generated_ids_list = []

# Максимальное количество новых токенов для генерации
max_new_tokens = 512

# ID токена, означающего конец генерации. Для Qwen это <|im_end|>
# Мы получаем его из токенизатора, чтобы не указывать вручную
eos_token_id = tokenizer.eos_token_id

print("Начинаем генерацию в цикле...")

# Оборачиваем в torch.no_grad(), так как мы не обучаем модель, это экономит память и ускоряет вычисления
with torch.no_grad():
    # Цикл генерации по одному токену
    for _ in range(max_new_tokens):
        # Передаем текущую последовательность токенов в модель
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)

        # Получаем логиты (сырые предсказания) для всех токенов в словаре
        # Нас интересуют только логиты для ПОСЛЕДНЕГО токена в последовательности
        # Их форма: [batch_size, sequence_length, vocab_size]
        next_token_logits = outputs.logits[0, -1, :]

        # Greedy decoding: выбираем ID токена с самым высоким логитом (наиболее вероятный следующий токен)
        next_token_id = torch.argmax(next_token_logits, dim=-1)

        # Добавляем ID нового токена в наш список сгенерированных
        generated_ids_list.append(next_token_id.item())

        # Проверяем, не является ли новый токен токеном конца последовательности
        if next_token_id.item() == eos_token_id:
            print("Найден токен конца последовательности. Завершение генерации.")
            break

        # Подготавливаем входные данные для следующего шага:
        # добавляем предсказанный токен к нашей последовательности
        input_ids = torch.cat(
            [input_ids, next_token_id.unsqueeze(0).unsqueeze(0)], dim=-1
        )
        # Также расширяем attention_mask, добавляя 1 для нового токена
        attention_mask = torch.cat(
            [attention_mask, torch.ones((1, 1), device=model.device, dtype=torch.long)],
            dim=1,
        )

print("Генерация завершена.")

# Переменная `output_ids` теперь содержит то же, что и в оригинальном коде
output_ids = generated_ids_list

# --- 4. Парсинг "думающего" контента (без изменений) ---
try:
    # Ищем индекс токена </think> (ID 151668) с конца списка
    index = len(output_ids) - output_ids[::-1].index(151668)
except ValueError:
    # Если токен не найден, считаем, что "думающего" контента нет
    index = 0

thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip(
    "\n"
)
content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

print("\n--- Результаты ---")
print("thinking content:", thinking_content)
print("\ncontent:", content)
pass

Начинаем генерацию в цикле...
Найден токен конца последовательности. Завершение генерации.
Генерация завершена.

--- Результаты ---
thinking content: 

content: A large language model (LLM) is a type of artificial intelligence that can understand, generate, and respond to human language. It is designed to process and analyze text, allowing it to perform tasks such as writing, answering questions, and even creating content.


In [6]:
print(text)

<|im_start|>user
Give me a short introduction to large language model.<|im_end|>
<|im_start|>assistant
<think>

</think>




In [ ]:
from datasets import load_dataset

dataset = load_dataset("dim/open_orca_4475_DeepSeek-R1-Distill-Qwen-1.5B")["train"]
dataset

Generating train split: 100%|██████████| 4475/4475 [00:00<00:00, 101047.16 examples/s]


Dataset({
    features: ['question', 'answer'],
    num_rows: 4475
})

In [9]:
dataset["question"][0]

'NEW: Peterson to media on handcuffs, chains: "I got the bling. Can\'t complain" Drew Peterson arrested in the death of his third wife, Kathleen Savio. Renewed interest in Savio\'s death came after Peterson\'s fourth wife disappeared. Peterson, through his attorney, denies any wrongdoing in either case.\n\nWrite an article based on these highlights.'

In [ ]:
# TODO: ембединги и голову зафиксировать(как в VAE), к модели добавить лору, обучить на flow matching. учить 